# H100 Tracker Tuning Notebook

Bu notebook tek iş için var: `scripts/tune_all_params_canary.py` scriptini H100 üzerinde düzenli şekilde çalıştırmak.

Kullanım sırası:

1. **Ayarlar** hücresinde `REPO_PATH` doğru mu bak.
2. **Preflight** hücresini çalıştır.
3. **Install + Build** hücrelerini çalıştır.
4. Önce **Smoke Run** yap.
5. Smoke temizse `RUN_FULL = True` yapıp **Full Canary Run** çalıştır.
6. En iyi config `cache/optuna_studies/*_best.yaml` olarak çıkar.

Not: Varsayılan `USE_SGLA=True`. H100 ortamında TensorRT engine taşınabilir olmayabileceği için PyTorch/SGLATrack daha güvenli başlangıçtır.


## 1. Ayarlar

Burayı genelde sadece bir kez değiştirmen yeterli.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Repo H100 makinede neredeyse burayı değiştir.
# Notebook repo içinden açıldıysa Path.cwd() çoğu zaman yeterli olur.
REPO_PATH = os.environ.get("TRACKER_REPO", str(Path.cwd()))

# H100 için güvenli varsayılan: PyTorch/SGLATrack.
# TensorRT kullanmak istiyorsan H100 üzerinde engine yeniden export edilmiş olmalı.
USE_SGLA = True

# Önce smoke. Full run'u smoke temizse aç.
RUN_SMOKE = True
RUN_FULL = False

SMOKE_TRIALS = 2
FULL_TRIALS = 100

STUDY_NAME = "all_params_h100_v1"
STUDY_DB = "cache/optuna_studies/all_params_h100_v1.db"
CHECKPOINT_DIR = "cache/optuna_studies"

SMOKE_SEQS = [
    "dataset5/uav4",
    "dataset3/truck_night",
    "dataset3/car8",
]

# Boş kalırsa scriptin kendi 14'lü canary seti kullanılır.
FULL_SEQS = []

repo = Path(REPO_PATH).expanduser().resolve()
print("Repo:", repo)
print("Backend:", "SGLA/PyTorch" if USE_SGLA else "TensorRT")


## 2. Yardımcı Komut Fonksiyonu

In [ ]:
assert repo.exists(), f"Repo bulunamadı: {repo}"
os.chdir(repo)


def run(cmd: str, check: bool = True):
    print(f"\n$ {cmd}")
    proc = subprocess.run(
        cmd,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    output = proc.stdout or ""
    print(output[-16000:])
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed ({proc.returncode}): {cmd}")
    return proc


def seq_args(seqs):
    return " ".join(f"--seq {s}" for s in seqs)

backend_arg = "--use-sgla" if USE_SGLA else ""


## 3. Preflight

GPU, Python, repo dosyaları ve checkpoint var mı bakıyoruz.


In [ ]:
run("pwd")
run("nvidia-smi || true", check=False)
run(f"{sys.executable} --version")

required = [
    "scripts/tune_all_params_canary.py",
    "configs/i12_rescue_area_gate.yaml",
    "data/contest_release/metadata/contestant_manifest.json",
    "models/SGLATrack/checkpoints/sglatrack_ep0297.pth.tar",
]

missing = []
for rel in required:
    p = repo / rel
    ok = p.exists()
    print(f"{rel:70s} {'OK' if ok else 'MISSING'}")
    if not ok:
        missing.append(rel)

if missing:
    raise FileNotFoundError("Eksik dosyalar: " + ", ".join(missing))


## 4. Install

`requirements.txt` Python tarafını kurar. TensorRT hâlâ CUDA/driver ile eşleşen sistem paketi olarak gelmeli.


In [ ]:
run(f"{sys.executable} -m pip install -q -r requirements.txt")
run(f"{sys.executable} - <<'PY'\nimport optuna, cv2, yaml, torch\nprint('optuna', optuna.__version__)\nprint('cv2', cv2.__version__)\nprint('torch', torch.__version__)\nPY")


## 5. Build `tracker_cpp`

Kalman/IMM/GMC C++ binding için gerekli.


In [ ]:
run("cmake -B build2 -DCMAKE_BUILD_TYPE=Release")
run("cmake --build build2 -j$(nproc) --target tracker_cpp")
run(f"{sys.executable} -m py_compile scripts/tune_all_params_canary.py")


## 6. Smoke Run

Önce küçük koşu. Bu patlarsa full run'a geçme.


In [ ]:
if RUN_SMOKE:
    smoke_cmd = (
        f"{sys.executable} scripts/tune_all_params_canary.py "
        f"{backend_arg} --n-trials {SMOKE_TRIALS} "
        f"--study-name {STUDY_NAME}_smoke "
        f"--study-db cache/optuna_studies/{STUDY_NAME}_smoke.db "
        f"--checkpoint-dir {CHECKPOINT_DIR} "
        f"{seq_args(SMOKE_SEQS)}"
    )
    run(smoke_cmd)
else:
    print("RUN_SMOKE=False, smoke atlandı.")


## 7. Full Canary Run

Smoke temizse ilk hücrede `RUN_FULL = True` yap ve bu hücreyi çalıştır.


In [ ]:
if RUN_FULL:
    full_seq_arg = seq_args(FULL_SEQS) if FULL_SEQS else ""
    full_cmd = (
        f"{sys.executable} scripts/tune_all_params_canary.py "
        f"{backend_arg} --n-trials {FULL_TRIALS} "
        f"--study-name {STUDY_NAME} "
        f"--study-db {STUDY_DB} "
        f"--checkpoint-dir {CHECKPOINT_DIR} "
        f"{full_seq_arg}"
    )
    run(full_cmd)
else:
    print("RUN_FULL=False. Smoke sonucunu gördükten sonra ilk hücreden True yap.")


## 8. Çıktılar

Best YAML ve summary burada görünür. Bunları local repo'ya alıp full 255 eval koşacağız.


In [ ]:
run("ls -lh cache/optuna_studies | tail -40", check=False)
print("\nBeklenen dosyalar:")
print(f"  {CHECKPOINT_DIR}/{STUDY_NAME}_smoke_best.yaml")
print(f"  {CHECKPOINT_DIR}/{STUDY_NAME}_best.yaml")
print(f"  {CHECKPOINT_DIR}/{STUDY_NAME}_summary.txt")


## 9. Best Config ile Hızlı Doğrulama

Full run bittikten sonra istersen best YAML ile küçük bir `ab_test` doğrulaması yap.


In [ ]:
BEST_CONFIG = Path(CHECKPOINT_DIR) / f"{STUDY_NAME}_best.yaml"

if BEST_CONFIG.exists():
    validate_cmd = (
        f"{sys.executable} scripts/ab_test.py "
        f"--imm --gmc --adaptive-r --mode ai_lead "
        f"--imm-config {BEST_CONFIG} "
        f"--seq dataset5/uav4 --seq dataset3/truck_night --seq dataset3/car8"
    )
    run(validate_cmd, check=False)
else:
    print(f"Best config henüz yok: {BEST_CONFIG}")
